# Cointegration Results: Filter & Sort

Loads the final cointegration CSV and lets you **filter** and **sort** by any column, then view the result.

Columns: `ticker1`, `ticker2`, `coint_t`, `pvalue`, `crit_1pct`, `crit_5pct`, `crit_10pct`, `cointegrated_5pct`, `collinear`, `spread_std`.

In [4]:
from pathlib import Path
import pandas as pd

# Resolve project root (same as cointegration_analysis.ipynb)
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / ".git").exists():
    _root = _root.parent

PROCESSED_DIR = _root / "research" / "processed"

# Load final result CSV (5% cointegrated, not collinear). Change filename to use full results.
csv_path = PROCESSED_DIR / "cointegration_results_5pct_noncollinear.csv"
df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} rows from {csv_path.name}")
print("Columns:", list(df.columns))
df.head()

Loaded 72,018 rows from cointegration_results_5pct_noncollinear.csv
Columns: ['ticker1', 'ticker2', 'coint_t', 'pvalue', 'crit_1pct', 'crit_5pct', 'crit_10pct', 'cointegrated_5pct', 'collinear', 'spread_std']


,ticker1,ticker2,coint_t,pvalue,crit_1pct,crit_5pct,crit_10pct,cointegrated_5pct,collinear,spread_std
0,A,ACAD,-3.681727,0.019340,-3.903570,-3.340103,-3.047207,True,False,15.782094
1,A,ACT,-3.627330,0.022657,-3.906343,-3.341645,-3.048277,True,False,13.267350
2,A,ALEX,-3.411092,0.041133,-3.903570,-3.340103,-3.047207,True,False,15.721292
3,A,APLE,-3.585391,0.025540,-3.903570,-3.340103,-3.047207,True,False,15.584666
4,A,ARI,-3.411684,0.041069,-3.903570,-3.340103,-3.047207,True,False,16.316412


## Filter and sort

Set the options below and run the cell. You can filter by any column and sort by any column.

In [5]:
# --- FILTER (optional) ---
# Leave filter_col empty string "" to skip filtering.
filter_col = ""           # e.g. "pvalue", "spread_std", "ticker1", "coint_t"
filter_op = "<"            # ">", "<", ">=", "<=", "==", "!=", "in" (for substring in str cols)
filter_val = 0.05          # number, True/False, or string (e.g. "XLE" for ticker)

# --- SORT ---
sort_col = "spread_std"   # any column name
sort_ascending = False     # False = descending (e.g. largest spread_std first)

# --- MAX ROWS TO SHOW ---
max_display = 500          # cap displayed rows

# --- APPLY ---
out = df.copy()

if filter_col and filter_col in out.columns:
    col = out[filter_col]
    if filter_op == "in" and col.dtype == object:
        out = out[col.astype(str).str.contains(str(filter_val), case=False, na=False)]
    elif filter_op == "==":
        out = out[col == filter_val]
    elif filter_op == "!=":
        out = out[col != filter_val]
    elif filter_op == ">":
        out = out[col > filter_val]
    elif filter_op == "<":
        out = out[col < filter_val]
    elif filter_op == ">=":
        out = out[col >= filter_val]
    elif filter_op == "<=":
        out = out[col <= filter_val]
    else:
        print(f"Unknown operator: {filter_op}")
    print(f"Filter: {filter_col} {filter_op} {filter_val} -> {len(out):,} rows")
else:
    if filter_col:
        print(f"No filter applied (column '{filter_col}' not found or empty).")

out = out.sort_values(sort_col, ascending=sort_ascending, na_position='last')
print(f"Sort: {sort_col} ({"ascending" if sort_ascending else "descending"})")
print(f"Showing up to {max_display} rows.")

display(out.head(max_display))

Sort: spread_std (descending)
Showing up to 500 rows.


,ticker1,ticker2,coint_t,pvalue,crit_1pct,crit_5pct,crit_10pct,cointegrated_5pct,collinear,spread_std
71305,UVXY,WU,-3.381187,0.044483,-3.90357,-3.340103,-3.047207,True,False,27351.060882
71282,UVXY,VIR,-3.384197,0.044136,-3.90357,-3.340103,-3.047207,True,False,27297.698402
71293,UVXY,VTRS,-3.541557,0.028885,-3.90357,-3.340103,-3.047207,True,False,26755.260620
71302,UVXY,WRLD,-3.341717,0.049249,-3.90357,-3.340103,-3.047207,True,False,26264.637919
71308,UVXY,XHR,-3.404402,0.041864,-3.90357,-3.340103,-3.047207,True,False,25878.901399
...,...,...,...,...,...,...,...,...,...,...
68565,SPXU,SRE,-3.842904,0.011871,-3.90357,-3.340103,-3.047207,True,False,341.182449
21270,BKNG,HWKN,-3.659375,0.020648,-3.90357,-3.340103,-3.047207,True,False,341.109750
68732,SPXU,WAB,-3.815216,0.012936,-3.90357,-3.340103,-3.047207,True,False,340.123227
68675,SPXU,V,-3.652845,0.021044,-3.90358,-3.340108,-3.047211,True,False,340.008425


### Example settings

- **Strict p-value:** `filter_col="pvalue"`, `filter_op="<"`, `filter_val=0.01`  
- **High spread (more mean-reversion potential):** `sort_col="spread_std"`, `sort_ascending=False`  
- **Strongest cointegration (most negative coint_t):** `sort_col="coint_t"`, `sort_ascending=True`  
- **Ticker contains:** `filter_col="ticker1"`, `filter_op="in"`, `filter_val="X"`  
- **Multiple filters:** run the cell once, then change `out` to `out = out[ ... ]` in a new cell for a second filter, or combine conditions in one expression.

## Select pairs for backtesting (up to 10)

**How we pick pairs:** (1) Rank by **cointegration strength** (lower pvalue = more significant). (2) Keep **tradeable spread**: filter spread_std into a band (e.g. 5th–95th percentile) so the spread moves enough to trade but is not noise. (3) **Diversify**: greedily choose pairs so each ticker appears in at most 1–2 pairs. Run the cell below to get a shortlist and a PAIRS list for pairs_backtest.ipynb.

In [6]:
# --- SELECT UP TO N PAIRS FOR BACKTEST (diversified, strong cointegration) ---
# Uses df from the load cell above.

max_pairs = 10
max_ticker_uses = 2   # each ticker in at most this many selected pairs
pvalue_max = 0.05     # already 5% in this CSV; tighten to 0.01 for stricter
spread_std_lo = df["spread_std"].quantile(0.05)   # exclude very low spread
spread_std_hi = df["spread_std"].quantile(0.95)  # exclude very high (noisy)

candidates = df[
    (df["pvalue"] <= pvalue_max) &
    (df["spread_std"] >= spread_std_lo) &
    (df["spread_std"] <= spread_std_hi)
].copy()
candidates = candidates.sort_values("pvalue", ascending=True).reset_index(drop=True)

selected = []
ticker_counts = {}
for _, row in candidates.iterrows():
    if len(selected) >= max_pairs:
        break
    t1, t2 = row["ticker1"], row["ticker2"]
    c1 = ticker_counts.get(t1, 0)
    c2 = ticker_counts.get(t2, 0)
    if c1 < max_ticker_uses and c2 < max_ticker_uses:
        selected.append(row)
        ticker_counts[t1] = c1 + 1
        ticker_counts[t2] = c2 + 1

pairs_df = pd.DataFrame(selected)

# --- Load name & description from research/raw (stocks + etfs) ---
RAW_DIR = _root / "research" / "raw"
meta = pd.concat([
    pd.read_csv(RAW_DIR / "stocks.csv", usecols=["Ticker", "Full Name", "Short Description"]),
    pd.read_csv(RAW_DIR / "etfs.csv", usecols=["Ticker", "Full Name", "Short Description"]),
], ignore_index=True).drop_duplicates(subset=["Ticker"], keep="first")
ticker_to_name = meta.set_index("Ticker")["Full Name"].to_dict()
ticker_to_desc = meta.set_index("Ticker")["Short Description"].to_dict()

pairs_df["ticker1_name"] = pairs_df["ticker1"].map(ticker_to_name).fillna("")
pairs_df["ticker1_desc"] = pairs_df["ticker1"].map(ticker_to_desc).fillna("")
pairs_df["ticker2_name"] = pairs_df["ticker2"].map(ticker_to_name).fillna("")
pairs_df["ticker2_desc"] = pairs_df["ticker2"].map(ticker_to_desc).fillna("")

print(f"Selected {len(pairs_df)} pairs (pvalue <= {pvalue_max}, spread_std in [{spread_std_lo:.2f}, {spread_std_hi:.2f}]).")
display(pairs_df[["ticker1", "ticker1_name", "ticker1_desc", "ticker2", "ticker2_name", "ticker2_desc", "pvalue", "coint_t", "spread_std"]])

# Copy-paste into pairs_backtest.ipynb PAIRS config:
PAIRS = [(r["ticker1"], r["ticker2"]) for _, r in pairs_df.iterrows()]
print("\n# PAIRS for pairs_backtest.ipynb:")
print("PAIRS =", PAIRS)

Selected 10 pairs (pvalue <= 0.05, spread_std in [1.14, 41.06]).


,ticker1,ticker1_name,ticker1_desc,ticker2,ticker2_name,ticker2_desc,pvalue,coint_t,spread_std
0,PECO,Phillips Edison & Company,Retail REIT,VOO,Vanguard S&P 500 ETF,Tracks the S&P 500 large-cap US equity index,1.782003e-24,-13.617311,1.900481
1,PECO,Phillips Edison & Company,Retail REIT,TIP,iShares TIPS Bond ETF,Tracks US Treasury inflation-protected securities,3.655447e-23,-12.943343,2.351297
6,MCW,"Mister Car Wash, Inc.",Specialized Consumer Services,OXY,Occidental Petroleum Corporation,Oil and gas exploration and production,2.527812e-14,-9.210107,1.510241
8,PSKY,Paramount Skydance Corporation,Entertainment,SNCY,Sun Country Airlines,Airline,1.044554e-13,-8.969015,4.905483
9,BNO,United States Brent Oil Fund,Tracks Brent crude oil futures,USO,United States Oil Fund,Tracks WTI crude oil futures,2.856708e-13,-8.797995,3.093469
10,EXE,Expand Energy Corp.,Oil and gas exploration and production,OXY,Occidental Petroleum Corporation,Oil and gas exploration and production,1.370188e-12,-8.530961,10.538321
11,LZ,"LegalZoom.com, Inc.",Research & Consulting Services,SPY,SPDR S&P 500 ETF Trust,Tracks the S&P 500 large-cap US equity index,3.183914e-12,-8.386910,3.729534
12,LZ,"LegalZoom.com, Inc.",Research & Consulting Services,RSP,Invesco S&P 500 Equal Weight ETF,Equal-weights S&P 500 constituents,3.189920e-12,-8.386587,3.732734
13,MCW,"Mister Car Wash, Inc.",Specialized Consumer Services,TIP,iShares TIPS Bond ETF,Tracks US Treasury inflation-protected securities,2.473521e-11,-8.034603,1.552465
18,SNCY,Sun Country Airlines,Airline,XLB,Materials Select Sector SPDR Fund,Tracks S&P 500 materials sector,4.831312e-09,-7.104427,3.177985



# PAIRS for pairs_backtest.ipynb:
PAIRS = [('PECO', 'VOO'), ('PECO', 'TIP'), ('MCW', 'OXY'), ('PSKY', 'SNCY'), ('BNO', 'USO'), ('EXE', 'OXY'), ('LZ', 'SPY'), ('LZ', 'RSP'), ('MCW', 'TIP'), ('SNCY', 'XLB')]
